# Experimento: preentrenamiento intermedio con AI4Privacy y WikiANN (español)

Compara dos NER que solo difieren en la etapa intermedia, y elige **solo con validación** (MEDDOCAN dev y sintético dev). Los conjuntos de prueba no se usan aquí.

- **B0**: `dccuchile/bert-base-spanish-wwm-cased` → MEDDOCAN train.
- **B1**: `dccuchile/bert-base-spanish-wwm-cased` → AI4Privacy-es (PER, LOC, ADDRESS) + WikiANN-es (PER, ORG, LOC) → MEDDOCAN train.

Ninguno usa el sintético dev para entrenar, porque comparte valores con el sintético test. Necesita GPU (Entorno de ejecución > Cambiar tipo > T4).

## 1. Subir el código
Sube `modelo-ner-v4.zip`.

In [ ]:
from google.colab import files
import zipfile, os
subido = files.upload()
zipfile.ZipFile(next(iter(subido))).extractall('/content')
os.chdir('/content/modelo-ner')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Dependencias

In [ ]:
for p in ["datasets", "accelerate", "presidio-analyzer", "presidio-anonymizer"]:
    !pip install -q {p}
!python -m spacy download es_core_news_sm -q

## 3. Convertir AI4Privacy (español) y WikiANN (español)
AI4Privacy: muestra de 20.000 textos del split train para entrenar y 2.000 para validar, con semilla fija. WikiANN: split train completo (20.000) y 2.000 del split validation; su split test no se usa. Cada script imprime el inventario de etiquetas.

In [ ]:
!python scripts/convertir_ai4privacy.py --n 20000 --n-val 2000 --salida datos/ai4privacy
!python scripts/convertir_wikiann.py --salida datos/wikiann

## 4. Etapa intermedia: AI4Privacy + WikiANN (unos 15 minutos)

In [ ]:
!python scripts/entrenar_ner.py --datos datos/ai4privacy/entrenamiento.jsonl datos/wikiann/entrenamiento.jsonl --val datos/ai4privacy/validacion.jsonl datos/wikiann/validacion.jsonl --base dccuchile/bert-base-spanish-wwm-cased --salida modelos/ner-bert-ai4p-etapa1 --epocas 2 --batch 16

## 5. Ajuste con MEDDOCAN: B0 (sin etapa intermedia) y B1 (con ella)
Cada uno tarda unos 15 a 25 minutos.

In [ ]:
!python scripts/entrenar_ner.py --datos datos/meddocan/train.jsonl --val datos/meddocan/dev.jsonl --base dccuchile/bert-base-spanish-wwm-cased --salida modelos/ner-bert-b0 --epocas 4 --batch 16
!python scripts/entrenar_ner.py --datos datos/meddocan/train.jsonl --val datos/meddocan/dev.jsonl --base modelos/ner-bert-ai4p-etapa1 --salida modelos/ner-bert-b1 --epocas 4 --batch 16

## 6. Comparación en validación (sin tocar los conjuntos de prueba)

In [ ]:
for m in ['b0', 'b1']:
    print('=====', m, '— MEDDOCAN dev (reporte del entrenamiento)')
    print(open(f'modelos/ner-bert-{m}/reporte_validacion.txt').read())
    !python scripts/evaluar_deteccion.py --datos datos/sintetico/dev.jsonl --nombre sintetico_dev_{m} --ner-backend hf --ner-modelo modelos/ner-bert-{m} --sin-presidio
    !python scripts/evaluar_deteccion.py --datos datos/meddocan/dev.jsonl --nombre meddocan_dev_{m} --ner-backend hf --ner-modelo modelos/ner-bert-{m} --sin-presidio

## 7. Descargar resultados
Guarda también los dos modelos en Drive: el que gane se usa después para el ajuste con la anotación de Doccano.

In [ ]:
!zip -qr resultados_ai4p.zip resultados/sintetico_dev_b0 resultados/sintetico_dev_b1 resultados/meddocan_dev_b0 resultados/meddocan_dev_b1 modelos/ner-bert-ai4p-etapa1/reporte_validacion.txt modelos/ner-bert-b0/reporte_validacion.txt modelos/ner-bert-b1/reporte_validacion.txt datos/ai4privacy/resumen.json datos/wikiann/resumen.json -x '*/salidas_*'
files.download('resultados_ai4p.zip')
from google.colab import drive; drive.mount('/content/drive')
!for m in ner-bert-b0 ner-bert-b1; do rsync -a --exclude checkpoints modelos/$m /content/drive/MyDrive/modelos-ner/; done